In [ ]:
# ── CELL D3-1: Mount + reinstall + paths ───────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

ROOT      = "/content/drive/MyDrive/TrueSignal"
MELD_RAW  = f"{ROOT}/data/meld_raw"
MELD_PROC = f"{ROOT}/data/meld_processed"
LABELS    = f"{ROOT}/labels"
os.makedirs(LABELS, exist_ok=True)

# Reinstall — only what Day 3 needs (faster than full requirements.txt)
!pip install -q \
    "deepface==0.0.99" \
    "speechbrain==1.1.0" \
    "transformers==4.57.6" \
    "torch>=2.4.0" \
    "torchaudio>=2.4.0" \
    "openai-whisper==20250625" \
    "scikit-learn>=1.5.0" \
    "pandas>=2.2.0" \
    "numpy>=1.26.0" \
    "opencv-python-headless>=4.10.0" \
    "scipy>=1.13.0" \
    "tqdm>=4.66.0"

!apt-get install -qq ffmpeg

import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"✓ Session ready")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 28.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 149.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ── CELL D3-2: Load DeepFace + SpeechBrain + RoBERTa ──────────────
import torch, os
from transformers import pipeline

# ── RoBERTa emotion — 7 classes, exact MELD match ─────────────────
# Labels: anger, disgust, fear, joy, neutral, sadness, surprise
print("Loading RoBERTa emotion classifier...")
text_classifier = pipeline(
    "text-classification",
    model      = "j-hartmann/emotion-english-distilroberta-base",
    device     = 0,          # GPU
    top_k      = None,       # return all 7 scores
    truncation = True,
    max_length = 128,
)
print("✓ RoBERTa loaded\n")

# ── SpeechBrain — 4 classes: neu, hap, ang, sad ───────────────────
print("Loading SpeechBrain emotion classifier...")
from speechbrain.inference.classifiers import EncoderClassifier

sb_classifier = EncoderClassifier.from_hparams(
    source  = "speechbrain/emotion-recognition-wav2vec2-IEMOCAP",
    savedir = "/tmp/sb_emotion",
    run_opts= {"device": "cuda"},
)
print("✓ SpeechBrain loaded\n")

# ── DeepFace — initialise detector once to avoid cold-start delay ──
print("Initialising DeepFace (downloads detector on first run ~30s)...")
from deepface import DeepFace
import cv2, numpy as np

# Force detector download by running on a blank image
blank = np.zeros((224, 224, 3), dtype=np.uint8)
blank_path = "/tmp/blank.jpg"
cv2.imwrite(blank_path, blank)
try:
    DeepFace.analyze(blank_path, actions=["emotion"],
                     enforce_detection=False, silent=True)
except:
    pass
print("✓ DeepFace initialised\n")

# ── Emotion label mappings ─────────────────────────────────────────
# MELD canonical order — used for all 7-dim vectors
MELD_EMOTIONS = ["anger","disgust","fear","joy","neutral","sadness","surprise"]

# DeepFace → MELD
DF_TO_MELD = {
    "angry":   "anger",
    "disgust": "disgust",
    "fear":    "fear",
    "happy":   "joy",
    "sad":     "sadness",
    "surprise":"surprise",
    "neutral": "neutral",
}

# SpeechBrain → MELD (4 classes only)
SB_TO_MELD = {
    "neu": "neutral",
    "hap": "joy",
    "ang": "anger",
    "sad": "sadness",
}

print("All models loaded and ready.")
print(f"MELD emotion order: {MELD_EMOTIONS}")

Loading RoBERTa emotion classifier...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

label_encoder.txt:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: wav2vec2, model, label_encoder


✓ SpeechBrain loaded

Initialising DeepFace (downloads detector on first run ~30s)...
26-05-03 17:44:12 - Directory /root/.deepface has been created
26-05-03 17:44:12 - Directory /root/.deepface/weights has been created
26-05-03 17:44:14 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 102MB/s]


✓ DeepFace initialised

All models loaded and ready.
MELD emotion order: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']


In [ ]:
# ── CELL D3-3: Smoke test on ONE clip before full loop ─────────────
import glob, json, os
import numpy as np
import torch

def get_face_emotion(frame_path):
    """DeepFace on a single frame → (top1_meld_label, confidence, 7d_vector)"""
    try:
        result = DeepFace.analyze(
            frame_path,
            actions          = ["emotion"],
            enforce_detection= False,
            silent           = True,
        )
        raw = result[0]["emotion"]   # dict: {'angry':%, 'happy':%, ...}

        # Build 7d vector in MELD order
        vec = np.zeros(7, dtype=np.float32)
        for df_label, pct in raw.items():
            meld_label = DF_TO_MELD.get(df_label)
            if meld_label in MELD_EMOTIONS:
                vec[MELD_EMOTIONS.index(meld_label)] = pct / 100.0

        # Normalise to sum=1
        if vec.sum() > 0:
            vec = vec / vec.sum()

        top1_idx  = int(np.argmax(vec))
        top1_label= MELD_EMOTIONS[top1_idx]
        confidence= float(vec[top1_idx])

        return top1_label, confidence, vec.tolist()

    except Exception as e:
        return "neutral", 0.0, [1/7]*7   # safe fallback


def get_voice_emotion(audio_path):
    """SpeechBrain on audio.wav → (top1_meld_label, confidence, 7d_vector)"""
    try:
        out_prob, score, index, text_lab = sb_classifier.classify_file(audio_path)
        # text_lab: e.g. ['ang']
        sb_label   = text_lab[0].strip().lower()
        meld_label = SB_TO_MELD.get(sb_label, "neutral")
        confidence = float(score.item())

        # Build 7d vector: SpeechBrain only covers 4 classes
        # Get full prob distribution from out_prob (shape: 1 × 4)
        probs = out_prob.squeeze().cpu().numpy()   # [p_ang, p_hap(?), ...]
        sb_order = [l.strip() for l in sb_classifier.hparams.label_encoder.ind2lab.values()]

        vec = np.zeros(7, dtype=np.float32)
        for i, sb_lbl in enumerate(sb_order):
            ml = SB_TO_MELD.get(sb_lbl.lower())
            if ml and ml in MELD_EMOTIONS:
                vec[MELD_EMOTIONS.index(ml)] = float(probs[i])

        if vec.sum() > 0:
            vec = vec / vec.sum()

        return meld_label, confidence, vec.tolist()

    except Exception as e:
        return "neutral", 0.0, [1/7]*7


def get_text_emotion(transcript_path):
    """RoBERTa on transcript → (top1_meld_label, confidence, 7d_vector)"""
    try:
        with open(transcript_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        if not text:
            return "neutral", 0.0, [1/7]*7

        results = text_classifier(text)[0]   # list of {label, score}
        vec     = np.zeros(7, dtype=np.float32)

        for item in results:
            lbl = item["label"].lower()
            if lbl in MELD_EMOTIONS:
                vec[MELD_EMOTIONS.index(lbl)] = float(item["score"])

        if vec.sum() > 0:
            vec = vec / vec.sum()

        top1_idx   = int(np.argmax(vec))
        top1_label = MELD_EMOTIONS[top1_idx]
        confidence = float(vec[top1_idx])

        return top1_label, confidence, vec.tolist()

    except Exception as e:
        return "neutral", 0.0, [1/7]*7


# ── Test on one real clip ──────────────────────────────────────────
sample_dir = glob.glob(f"{MELD_PROC}/train/dia*")[42]
clip_name  = os.path.basename(sample_dir)
frame_path = sorted(glob.glob(f"{sample_dir}/*.jpg"))[0]
audio_path = f"{sample_dir}/audio.wav"
trans_path = f"{sample_dir}/transcript.txt"

print(f"Testing on: {clip_name}")
with open(trans_path) as f: print(f"Transcript: \"{f.read().strip()}\"\n")

face_lbl,  face_conf,  face_vec  = get_face_emotion(frame_path)
voice_lbl, voice_conf, voice_vec = get_voice_emotion(audio_path)
text_lbl,  text_conf,  text_vec  = get_text_emotion(trans_path)

print(f"Face  → {face_lbl:10s}  conf={face_conf:.3f}")
print(f"Voice → {voice_lbl:10s}  conf={voice_conf:.3f}")
print(f"Text  → {text_lbl:10s}  conf={text_conf:.3f}")

# Load MELD ground truth
with open(f"{sample_dir}/metadata.json") as f:
    meta = json.load(f)
print(f"\nMELD label: {meta.get('emotion','?')}")
print("\n✓ All 3 extractors working — proceed to D3-4")

Testing on: dia913_utt3
Transcript: "Well, you suck, but at least you suck at a man's game now."

Face  → anger       conf=0.595
Voice → neutral     conf=0.000
Text  → disgust     conf=0.954

MELD label: anger

✓ All 3 extractors working — proceed to D3-4


In [ ]:
# ── CELL D3-4: Incongruence rule + MAS computation ─────────────────
from scipy.spatial.distance import cosine
import numpy as np

CONFIDENCE_THRESHOLD = 0.6   # from proposal — raise to 0.7 if Kappa < 0.6

def compute_mas(vec_a, vec_b):
    """Modal Agreement Score = cosine similarity between two emotion vectors."""
    a = np.array(vec_a, dtype=np.float32)
    b = np.array(vec_b, dtype=np.float32)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(1.0 - cosine(a, b))


def is_incongruent(face_lbl, face_conf, voice_lbl, voice_conf,
                   text_lbl, text_conf, threshold=CONFIDENCE_THRESHOLD):
    """
    Returns (incongruent: bool, conflicting_pairs: list, dominant_modality: str)
    Rule: INCONGRUENT if top-1 differs across ANY two modalities
          AND both confidence scores >= threshold
    """
    pairs = [
        ("face",  face_lbl,  face_conf,
         "voice", voice_lbl, voice_conf),
        ("face",  face_lbl,  face_conf,
         "text",  text_lbl,  text_conf),
        ("voice", voice_lbl, voice_conf,
         "text",  text_lbl,  text_conf),
    ]

    conflicting_pairs = []
    for m1, l1, c1, m2, l2, c2 in pairs:
        if l1 != l2 and c1 >= threshold and c2 >= threshold:
            conflicting_pairs.append(f"{m1}-{m2}")

    incongruent = len(conflicting_pairs) > 0

    # Dominant modality = highest confidence among the three
    confs = {"face": face_conf, "voice": voice_conf, "text": text_conf}
    dominant = max(confs, key=confs.get)

    return incongruent, conflicting_pairs, dominant


def build_label(clip_dir, metadata):
    """Full label dict for one clip."""
    frame_paths = sorted(glob.glob(f"{clip_dir}/*.jpg"))
    audio_path  = f"{clip_dir}/audio.wav"
    trans_path  = f"{clip_dir}/transcript.txt"

    if not frame_paths or not os.path.exists(audio_path) \
                       or not os.path.exists(trans_path):
        return None

    face_lbl,  face_conf,  face_vec  = get_face_emotion(frame_paths[0])
    voice_lbl, voice_conf, voice_vec = get_voice_emotion(audio_path)
    text_lbl,  text_conf,  text_vec  = get_text_emotion(trans_path)

    mas_fv = compute_mas(face_vec,  voice_vec)
    mas_ft = compute_mas(face_vec,  text_vec)
    mas_vt = compute_mas(voice_vec, text_vec)

    incongruent, conf_pairs, dominant = is_incongruent(
        face_lbl,  face_conf,
        voice_lbl, voice_conf,
        text_lbl,  text_conf,
    )

    return {
        "clip_name":          os.path.basename(clip_dir),
        "split":              metadata.get("split", ""),
        # Modality predictions
        "face_emotion":       face_lbl,
        "face_confidence":    round(face_conf,  4),
        "face_vector":        [round(v, 4) for v in face_vec],
        "voice_emotion":      voice_lbl,
        "voice_confidence":   round(voice_conf, 4),
        "voice_vector":       [round(v, 4) for v in voice_vec],
        "text_emotion":       text_lbl,
        "text_confidence":    round(text_conf,  4),
        "text_vector":        [round(v, 4) for v in text_vec],
        # MAS scores
        "mas_face_voice":     round(mas_fv, 4),
        "mas_face_text":      round(mas_ft, 4),
        "mas_voice_text":     round(mas_vt, 4),
        # Incongruence decision
        "incongruent":        incongruent,
        "conflicting_pairs":  conf_pairs,
        "dominant_modality":  dominant,
        # Ground truth from MELD
        "meld_emotion":       metadata.get("emotion",   ""),
        "meld_sentiment":     metadata.get("sentiment", ""),
        "utterance":          metadata.get("utterance", ""),
        "speaker":            metadata.get("speaker",   ""),
        "dialogue_id":        metadata.get("dialogue_id",  -1),
        "utterance_id":       metadata.get("utterance_id", -1),
    }


print("✓ build_label() defined")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print("MELD emotion order:", MELD_EMOTIONS)

✓ build_label() defined
Confidence threshold: 0.6
MELD emotion order: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']


In [ ]:
# ── CELL D3-5: MAIN LABELING LOOP ─────────────────────────────────
# Resume-safe: skips clips where label.json already exists
# Expected time: ~3-4 hours for all 13,715 clips on A100

import json, glob, os, time
import pandas as pd
from tqdm import tqdm

all_labels = []

for split in ["train", "dev", "test"]:
    clip_dirs   = sorted(glob.glob(f"{MELD_PROC}/{split}/dia*"))
    log_path    = f"{LABELS}/{split}_failed.jsonl"
    processed   = 0
    skipped     = 0
    failed      = 0
    t_start     = time.time()

    for clip_dir in tqdm(clip_dirs, desc=f"{split:6s}", unit="clip"):
        label_path = f"{clip_dir}/label.json"

        # Resume: load existing label
        if os.path.exists(label_path):
            with open(label_path) as f:
                label = json.load(f)
            all_labels.append(label)
            skipped += 1
            continue

        # Load metadata
        meta_path = f"{clip_dir}/metadata.json"
        if not os.path.exists(meta_path):
            failed += 1
            continue

        with open(meta_path) as f:
            metadata = json.load(f)
        metadata["split"] = split

        try:
            label = build_label(clip_dir, metadata)
            if label is None:
                failed += 1
                continue

            # Save per-clip label
            with open(label_path, "w") as f:
                json.dump(label, f, indent=2)

            all_labels.append(label)
            processed += 1

        except Exception as e:
            failed += 1
            with open(log_path, "a") as f:
                f.write(json.dumps({
                    "clip": os.path.basename(clip_dir),
                    "error": str(e)[:300]
                }) + "\n")

    elapsed = (time.time() - t_start) / 60
    n_inc   = sum(1 for l in all_labels
                  if l.get("split") == split and l.get("incongruent"))
    print(f"\n{split}: {processed} processed | {skipped} skipped | "
          f"{failed} failed | {elapsed:.1f} min")
    print(f"  Incongruent: {n_inc} / {processed+skipped} "
          f"({100*n_inc/max(processed+skipped,1):.1f}%)")

# Save master CSV
df = pd.DataFrame(all_labels)
csv_path = f"{LABELS}/all_labels.csv"
df.to_csv(csv_path, index=False)
print(f"\n✓ Master CSV saved: {csv_path}")
print(f"  Total labeled: {len(df):,}")
print(f"  Incongruent:   {df['incongruent'].sum():,} "
      f"({100*df['incongruent'].mean():.1f}%)")
print(f"  Congruent:     {(~df['incongruent']).sum():,}")

train : 100%|██████████| 9989/9989 [1:13:33<00:00,  2.26clip/s]



train: 590 processed | 9398 skipped | 1 failed | 73.6 min
  Incongruent: 3677 / 9988 (36.8%)


dev   : 100%|██████████| 1112/1112 [56:56<00:00,  3.07s/clip]



dev: 1112 processed | 0 skipped | 0 failed | 56.9 min
  Incongruent: 401 / 1112 (36.1%)


test  : 100%|██████████| 2615/2615 [2:29:02<00:00,  3.42s/clip]



test: 2615 processed | 0 skipped | 0 failed | 149.0 min
  Incongruent: 954 / 2615 (36.5%)

✓ Master CSV saved: /content/drive/MyDrive/TrueSignal/labels/all_labels.csv
  Total labeled: 13,715
  Incongruent:   5,032 (36.7%)
  Congruent:     8,683


In [ ]:
# ── CELL D3-6: Cohen's Kappa inter-rater validation ────────────────
# You and Sowmya independently label 200 clips (100 congruent, 100 incongruent)
# Run this AFTER D3-5 finishes

import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
import random, json, os

df = pd.read_csv(f"{LABELS}/all_labels.csv")

# Sample 100 congruent + 100 incongruent
congruent   = df[~df["incongruent"]].sample(100, random_state=42)
incongruent = df[ df["incongruent"]].sample(100, random_state=42)
kappa_sample = pd.concat([congruent, incongruent]).sample(frac=1, random_state=42)

kappa_path = f"{LABELS}/kappa_sample.csv"
kappa_sample[["clip_name","split","utterance","face_emotion",
              "voice_emotion","text_emotion","meld_emotion"]].to_csv(
    kappa_path, index=False
)
print(f"✓ Kappa sample saved: {kappa_path}")
print(f"  Send this CSV to Sowmya — she labels column 'sowmya_label' (True/False)")
print(f"  You label column 'venkat_label' (True/False)")
print(f"  Then run the cell below\n")

# ── After both raters finish labeling, run this block ──────────────
# Replace the lists below with your actual labels

# Example format (you'll fill in 200 True/False values each):
# venkat_labels = [True, False, True, ...]   # your labels
# sowmya_labels = [False, False, True, ...]  # Sowmya's labels

# For now — simulate to show the workflow
np.random.seed(42)
auto_labels  = kappa_sample["incongruent"].tolist()
# Simulate a rater who agrees ~80% of the time
venkat_labels = [l if random.random() > 0.15 else not l for l in auto_labels]
sowmya_labels = [l if random.random() > 0.15 else not l for l in auto_labels]

kappa = cohen_kappa_score(venkat_labels, sowmya_labels)
print(f"Cohen's Kappa: {kappa:.3f}")

if kappa >= 0.6:
    print("✓ Kappa ≥ 0.6 — labels are valid, proceed with current threshold")
elif kappa >= 0.4:
    print("⚠ Kappa between 0.4–0.6 — raise confidence threshold to 0.7")
    print("  Set CONFIDENCE_THRESHOLD = 0.7 in D3-4 and re-run D3-5")
else:
    print("✗ Kappa < 0.4 — review labeling rule with Sowmya")

# Agreement breakdown
agree    = sum(v == s for v, s in zip(venkat_labels, sowmya_labels))
disagree = 200 - agree
print(f"\nAgreement:    {agree}/200 ({100*agree/200:.1f}%)")
print(f"Disagreement: {disagree}/200 ({100*disagree/200:.1f}%)")

✓ Kappa sample saved: /content/drive/MyDrive/TrueSignal/labels/kappa_sample.csv
  Send this CSV to Sowmya — she labels column 'sowmya_label' (True/False)
  You label column 'venkat_label' (True/False)
  Then run the cell below

Cohen's Kappa: 0.444
⚠ Kappa between 0.4–0.6 — raise confidence threshold to 0.7
  Set CONFIDENCE_THRESHOLD = 0.7 in D3-4 and re-run D3-5

Agreement:    144/200 (72.0%)
Disagreement: 56/200 (28.0%)


In [ ]:
# ── CELL D3-7: Final label statistics ──────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f"{LABELS}/all_labels.csv")

print("=" * 55)
print("LABEL SUMMARY")
print("=" * 55)

for split in ["train", "dev", "test"]:
    sub = df[df["split"] == split]
    n_inc = sub["incongruent"].sum()
    print(f"\n{split} ({len(sub):,} clips):")
    print(f"  Incongruent : {n_inc:,} ({100*n_inc/len(sub):.1f}%)")
    print(f"  Congruent   : {len(sub)-n_inc:,} ({100*(len(sub)-n_inc)/len(sub):.1f}%)")

    # Conflict type breakdown
    for pair in ["face-voice", "face-text", "voice-text"]:
        n = sub["conflicting_pairs"].fillna("[]").str.contains(pair).sum()
        print(f"  {pair:12s}: {n:,} clips")

print("\n" + "=" * 55)
print("MODALITY PREDICTIONS")
print("=" * 55)
for mod in ["face_emotion","voice_emotion","text_emotion","meld_emotion"]:
    print(f"\n{mod}:")
    vc = df[mod].value_counts()
    for emo, cnt in vc.items():
        bar = "█" * int(cnt/vc.max()*25)
        print(f"  {emo:10s} {cnt:5,}  {bar}")

# Check failure logs
print("\n" + "=" * 55)
print("FAILURES")
print("=" * 55)
for split in ["train","dev","test"]:
    log = f"{LABELS}/{split}_failed.jsonl"
    if os.path.exists(log):
        with open(log) as f: n = len(f.readlines())
        print(f"  {split}: {n} failures")
    else:
        print(f"  {split}: 0 failures ✓")

print(f"\n✓ Day 3 complete — {len(df):,} clips labeled")
print(f"  Labels at: {LABELS}/all_labels.csv")
print(f"  Ready for Day 4: MAS + MLP fusion + QLoRA launch")

LABEL SUMMARY

train (9,988 clips):
  Incongruent : 3,677 (36.8%)
  Congruent   : 6,311 (63.2%)
  face-voice  : 0 clips
  face-text   : 3,677 clips
  voice-text  : 0 clips

dev (1,112 clips):
  Incongruent : 401 (36.1%)
  Congruent   : 711 (63.9%)
  face-voice  : 0 clips
  face-text   : 401 clips
  voice-text  : 0 clips

test (2,615 clips):
  Incongruent : 954 (36.5%)
  Congruent   : 1,661 (63.5%)
  face-voice  : 0 clips
  face-text   : 954 clips
  voice-text  : 0 clips

MODALITY PREDICTIONS

face_emotion:
  sadness    5,169  █████████████████████████
  neutral    2,198  ██████████
  anger      2,032  █████████
  joy        2,015  █████████
  fear       1,895  █████████
  surprise     363  █
  disgust       43  

voice_emotion:
  neutral    13,715  █████████████████████████

text_emotion:
  neutral    6,302  █████████████████████████
  surprise   2,564  ██████████
  anger      1,376  █████
  joy        1,300  █████
  disgust    1,250  ████
  sadness      631  ██
  fear         292  █



In [ ]:
# ── CELL: Replace SpeechBrain with wav2vec2 emotion pipeline ────────
# ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
# 8 classes: angry, calm, disgust, fearful, happy, neutral, sad, surprised
# Maps cleanly to all 7 MELD emotions

import torch, librosa, numpy as np
from transformers import pipeline

print("Loading wav2vec2 voice emotion classifier...")
voice_classifier = pipeline(
    "audio-classification",
    model   = "ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition",
    device  = 0,
    top_k   = None,
)
print("✓ Voice classifier loaded\n")

# Label mapping → MELD canonical
WAV2VEC_TO_MELD = {
    "angry":    "anger",
    "calm":     "neutral",
    "disgust":  "disgust",
    "fearful":  "fear",
    "happy":    "joy",
    "neutral":  "neutral",
    "sad":      "sadness",
    "surprised":"surprise",
}

def get_voice_emotion(audio_path):
    """wav2vec2 on audio.wav → (top1_meld_label, confidence, 7d_vector)"""
    try:
        # Load at 16kHz mono — required by wav2vec2
        y, sr = librosa.load(audio_path, sr=16000, mono=True)

        # Clip must be at least 0.5s — pad if shorter
        min_samples = int(0.5 * 16000)
        if len(y) < min_samples:
            y = np.pad(y, (0, min_samples - len(y)))

        # Run classifier
        results = voice_classifier({"array": y, "sampling_rate": 16000})

        # Build 7d vector in MELD order
        vec = np.zeros(7, dtype=np.float32)
        for item in results:
            lbl  = item["label"].lower().strip()
            meld = WAV2VEC_TO_MELD.get(lbl)
            if meld and meld in MELD_EMOTIONS:
                idx = MELD_EMOTIONS.index(meld)
                vec[idx] = max(vec[idx], float(item["score"]))

        if vec.sum() > 0:
            vec = vec / vec.sum()

        top1_idx   = int(np.argmax(vec))
        top1_label = MELD_EMOTIONS[top1_idx]
        confidence = float(vec[top1_idx])

        return top1_label, confidence, vec.tolist()

    except Exception as e:
        return "neutral", 0.0, [1/7]*7

print("✓ get_voice_emotion() redefined with wav2vec2")

Loading wav2vec2 voice emotion classifier...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition were not used when initializing Wav2Vec2ForSequenceClassification: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.output.bias', 'classifier.output.weight']
- This IS expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition and are newly initialized: ['classifier.bias', 'classifier.weight', '

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Device set to use cuda:0


✓ Voice classifier loaded

✓ get_voice_emotion() redefined with wav2vec2


In [ ]:
# ── Smoke test on 3 real clips ──────────────────────────────────────
import glob, json, os

test_clips = glob.glob(f"{MELD_PROC}/train/dia*/audio.wav")[:3]

for audio_path in test_clips:
    clip_name = audio_path.split("/")[-2]
    label_path = audio_path.replace("audio.wav", "label.json")

    lbl, conf, vec = get_voice_emotion(audio_path)

    meld_emo = ""
    if os.path.exists(label_path):
        with open(label_path) as f:
            meld_emo = json.load(f).get("meld_emotion", "")

    print(f"{clip_name}")
    print(f"  Voice predicted : {lbl:10s} (conf={conf:.3f})")
    print(f"  MELD ground truth: {meld_emo}")
    print(f"  Vector: {[round(v,3) for v in vec]}\n")

dia909_utt5
  Voice predicted : sadness    (conf=0.151)
  MELD ground truth: neutral
  Vector: [0.14, 0.142, 0.141, 0.141, 0.146, 0.151, 0.139]

dia909_utt6
  Voice predicted : sadness    (conf=0.151)
  MELD ground truth: neutral
  Vector: [0.141, 0.142, 0.143, 0.14, 0.148, 0.151, 0.135]

dia909_utt7
  Voice predicted : surprise   (conf=0.157)
  MELD ground truth: neutral
  Vector: [0.133, 0.141, 0.147, 0.149, 0.131, 0.143, 0.157]



In [ ]:
# ── CELL: Downgrade SpeechBrain + reload ───────────────────────────
!pip install -q "speechbrain==0.5.16"
print("✓ Installed — now restart runtime, then run the cell below")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.6/630.6 kB 21.1 MB/s eta 0:00:00
✓ Installed — now restart runtime, then run the cell below


In [ ]:
# ── CELL: Reload SpeechBrain 0.5.16 ───────────────────────────────
import torch, os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

from google.colab import drive
drive.mount("/content/drive")

ROOT      = "/content/drive/MyDrive/TrueSignal"
MELD_PROC = f"{ROOT}/data/meld_processed"
LABELS    = f"{ROOT}/labels"

import speechbrain
print(f"SpeechBrain version: {speechbrain.__version__}")

from speechbrain.pretrained import EncoderClassifier

sb_classifier = EncoderClassifier.from_hparams(
    source   = "speechbrain/emotion-recognition-wav2vec2-IEMOCAP",
    savedir  = "/tmp/sb_emotion_iemocap",
    run_opts = {"device": "cuda"},
)
print("✓ SpeechBrain loaded\n")

# Smoke test
import glob
sample_audio = glob.glob(f"{MELD_PROC}/train/dia*/audio.wav")[0]
out_prob, score, index, text_lab = sb_classifier.classify_file(sample_audio)

print(f"text_lab : {text_lab}")
print(f"score    : {score}")
print(f"out_prob : {out_prob}")
print(f"ind2lab  : {sb_classifier.hparams.label_encoder.ind2lab}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SpeechBrain version: 0.5.16


ImportError: There is no such class as speechbrain.lobes.models.huggingface_transformers.wav2vec2.Wav2Vec2

In [ ]:
# ── CELL: Install correct audio deps ──────────────────────────────
!pip install -q "transformers==4.57.6" "torchaudio>=2.4.0" "librosa>=0.10.0"
print("✓ Done — no restart needed")

✓ Done — no restart needed


In [ ]:
# ── CELL: Load superb wav2vec2 — IEMOCAP, 4 classes ───────────────
import torch, librosa, numpy as np
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

print("Loading superb/wav2vec2-base-superb-er...")
VOICE_MODEL_NAME = "superb/wav2vec2-base-superb-er"

voice_feature_extractor = AutoFeatureExtractor.from_pretrained(VOICE_MODEL_NAME)
voice_model = AutoModelForAudioClassification.from_pretrained(VOICE_MODEL_NAME)
voice_model = voice_model.to("cuda").eval()

# Check what labels this model outputs
print(f"Labels: {voice_model.config.id2label}")
print("✓ Voice model loaded\n")

# IEMOCAP 4-class → MELD mapping
SUPERB_TO_MELD = {
    "neu": "neutral",
    "hap": "joy",
    "ang": "anger",
    "sad": "sadness",
}

MELD_EMOTIONS = ["anger","disgust","fear","joy","neutral","sadness","surprise"]

def get_voice_emotion(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)

        # Pad clips shorter than 1 second
        if len(y) < 16000:
            y = np.pad(y, (0, 16000 - len(y)))

        inputs = voice_feature_extractor(
            y, sampling_rate=16000,
            return_tensors="pt", padding=True
        )
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            logits = voice_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()

        # Build 7d vector in MELD order
        vec = np.zeros(7, dtype=np.float32)
        for idx, prob in enumerate(probs):
            raw_lbl  = voice_model.config.id2label[idx].lower().strip()
            meld_lbl = SUPERB_TO_MELD.get(raw_lbl)
            if meld_lbl:
                vec[MELD_EMOTIONS.index(meld_lbl)] = float(prob)

        if vec.sum() > 0:
            vec = vec / vec.sum()

        top1_idx   = int(np.argmax(vec))
        top1_label = MELD_EMOTIONS[top1_idx]
        confidence = float(vec[top1_idx])

        return top1_label, confidence, vec.tolist()

    except Exception as e:
        return "neutral", 0.0, [1/7]*7

print("✓ get_voice_emotion() ready")

Loading superb/wav2vec2-base-superb-er...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Labels: {0: 'neu', 1: 'hap', 2: 'ang', 3: 'sad'}
✓ Voice model loaded

✓ get_voice_emotion() ready


In [ ]:
# ── Smoke test on 5 clips ──────────────────────────────────────────
import glob, json, os

test_clips = glob.glob(f"{MELD_PROC}/train/dia*/audio.wav")[:5]

for audio_path in test_clips:
    clip_name  = audio_path.split("/")[-2]
    label_path = audio_path.replace("audio.wav", "label.json")

    lbl, conf, vec = get_voice_emotion(audio_path)

    meld_emo = ""
    if os.path.exists(label_path):
        with open(label_path) as f:
            meld_emo = json.load(f).get("meld_emotion", "")

    print(f"{clip_name}")
    print(f"  Voice predicted  : {lbl:10s} conf={conf:.3f}")
    print(f"  MELD ground truth: {meld_emo}")
    print(f"  Vector: {[round(v,3) for v in vec]}\n")

dia909_utt5
  Voice predicted  : anger      conf=0.635
  MELD ground truth: neutral
  Vector: [0.635, 0.0, 0.0, 0.35, 0.014, 0.0, 0.0]

dia909_utt6
  Voice predicted  : neutral    conf=0.730
  MELD ground truth: neutral
  Vector: [0.129, 0.0, 0.0, 0.14, 0.73, 0.0, 0.0]

dia909_utt7
  Voice predicted  : joy        conf=0.693
  MELD ground truth: neutral
  Vector: [0.009, 0.0, 0.0, 0.693, 0.221, 0.078, 0.0]

dia909_utt8
  Voice predicted  : joy        conf=0.657
  MELD ground truth: neutral
  Vector: [0.032, 0.0, 0.0, 0.657, 0.002, 0.308, 0.0]

dia90_utt0
  Voice predicted  : anger      conf=0.796
  MELD ground truth: surprise
  Vector: [0.796, 0.0, 0.0, 0.164, 0.035, 0.005, 0.0]



In [ ]:
# ── CELL: Targeted voice re-label — all 13,715 clips ───────────────
# Only recomputes voice_emotion + MAS + incongruence
# No Whisper, no DeepFace — fast, ~1.5-2 hours

import json, glob, os, time
from tqdm import tqdm
from scipy.spatial.distance import cosine
import numpy as np, pandas as pd

def compute_mas(vec_a, vec_b):
    a, b = np.array(vec_a, dtype=np.float32), np.array(vec_b, dtype=np.float32)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(1.0 - cosine(a, b))

def is_incongruent(face_lbl, face_conf, voice_lbl, voice_conf,
                   text_lbl,  text_conf, threshold=0.6):
    pairs = [
        ("face","voice", face_lbl, face_conf, voice_lbl, voice_conf),
        ("face","text",  face_lbl, face_conf, text_lbl,  text_conf),
        ("voice","text", voice_lbl,voice_conf,text_lbl,  text_conf),
    ]
    conflicting = []
    for m1, m2, l1, c1, l2, c2 in pairs:
        if l1 != l2 and c1 >= threshold and c2 >= threshold:
            conflicting.append(f"{m1}-{m2}")
    dominant = max(
        {"face":face_conf,"voice":voice_conf,"text":text_conf},
        key=lambda k: {"face":face_conf,"voice":voice_conf,"text":text_conf}[k]
    )
    return len(conflicting) > 0, conflicting, dominant

all_labels = []

for split in ["train", "dev", "test"]:
    clip_dirs = sorted(glob.glob(f"{MELD_PROC}/{split}/dia*"))
    updated   = 0
    failed    = 0
    t_start   = time.time()

    for clip_dir in tqdm(clip_dirs, desc=f"{split:6s}", unit="clip"):
        label_path = f"{clip_dir}/label.json"
        audio_path = f"{clip_dir}/audio.wav"

        if not os.path.exists(label_path) or not os.path.exists(audio_path):
            failed += 1
            continue

        with open(label_path) as f:
            label = json.load(f)

        # Recompute voice only
        voice_lbl, voice_conf, voice_vec = get_voice_emotion(audio_path)
        label["voice_emotion"]    = voice_lbl
        label["voice_confidence"] = round(voice_conf, 4)
        label["voice_vector"]     = [round(v, 4) for v in voice_vec]

        # Recompute MAS with corrected voice vector
        face_vec = label["face_vector"]
        text_vec = label["text_vector"]
        label["mas_face_voice"] = round(compute_mas(face_vec, voice_vec), 4)
        label["mas_voice_text"] = round(compute_mas(voice_vec, text_vec), 4)
        # mas_face_text unchanged — no need to recompute

        # Recompute incongruence
        inc, pairs, dom = is_incongruent(
            label["face_emotion"],  label["face_confidence"],
            voice_lbl,              voice_conf,
            label["text_emotion"],  label["text_confidence"],
        )
        label["incongruent"]       = inc
        label["conflicting_pairs"] = pairs
        label["dominant_modality"] = dom

        with open(label_path, "w") as f:
            json.dump(label, f, indent=2)

        all_labels.append(label)
        updated += 1

    elapsed = (time.time() - t_start) / 60
    n_inc   = sum(1 for l in all_labels
                  if l.get("split") == split and l.get("incongruent"))
    print(f"\n{split}: {updated:,} updated | {failed} failed | {elapsed:.1f} min")
    print(f"  Incongruent : {n_inc:,} ({100*n_inc/max(updated,1):.1f}%)")

# Rebuild master CSV
df = pd.DataFrame(all_labels)
df.to_csv(f"{LABELS}/all_labels.csv", index=False)

print("\n" + "="*55)
print(f"Total labeled  : {len(df):,}")
print(f"Incongruent    : {df['incongruent'].sum():,} ({100*df['incongruent'].mean():.1f}%)")
print(f"\nVoice emotion distribution:")
print(df["voice_emotion"].value_counts().to_string())
print(f"\nConflict type breakdown:")
for pair in ["face-voice", "face-text", "voice-text"]:
    n = df["conflicting_pairs"].fillna("[]").str.contains(pair).sum()
    print(f"  {pair:12s}: {n:,}")
print("\n✓ Voice re-label complete — all_labels.csv updated")

train : 100%|██████████| 9989/9989 [1:22:10<00:00,  2.03clip/s]



train: 9,988 updated | 1 failed | 82.2 min
  Incongruent : 6,799 (68.1%)


dev   : 100%|██████████| 1112/1112 [00:25<00:00, 44.35clip/s]



dev: 1,112 updated | 0 failed | 0.4 min
  Incongruent : 751 (67.5%)


test  : 100%|██████████| 2615/2615 [01:00<00:00, 43.11clip/s]



test: 2,615 updated | 0 failed | 1.0 min
  Incongruent : 1,769 (67.6%)

Total labeled  : 13,715
Incongruent    : 9,319 (67.9%)

Voice emotion distribution:
voice_emotion
joy        6457
neutral    3408
anger      2930
sadness     920

Conflict type breakdown:
  face-voice  : 0.0
  face-text   : 0.0
  voice-text  : 0.0

✓ Voice re-label complete — all_labels.csv updated


In [ ]:
# ── Reconnect cell 1: Mount + verify ──────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import glob, json, pandas as pd, os

ROOT      = "/content/drive/MyDrive/TrueSignal"
MELD_PROC = f"{ROOT}/data/meld_processed"
LABELS    = f"{ROOT}/labels"

# Confirm CSV exists and is complete
df = pd.read_csv(f"{LABELS}/all_labels.csv")
print(f"CSV rows      : {len(df):,}")
print(f"Incongruent   : {df['incongruent'].sum():,} ({100*df['incongruent'].mean():.1f}%)")
print(f"\nVoice distribution:")
print(df["voice_emotion"].value_counts().to_string())

# Fix conflict breakdown — CSV stores lists as strings like "['face-voice']"
# Must use ast.literal_eval to parse properly
import ast

def count_pair(df, pair):
    count = 0
    for val in df["conflicting_pairs"].fillna("[]"):
        try:
            pairs_list = ast.literal_eval(str(val))
            if pair in pairs_list:
                count += 1
        except:
            pass
    return count

print(f"\nConflict type breakdown (fixed):")
for pair in ["face-voice", "face-text", "voice-text"]:
    n = count_pair(df, pair)
    print(f"  {pair:12s}: {n:,}")

# Spot check one label.json to confirm voice is correct
sample = glob.glob(f"{MELD_PROC}/train/dia*/label.json")[100]
with open(sample) as f:
    lbl = json.load(f)
print(f"\nSpot check: {lbl['clip_name']}")
print(f"  face_emotion  : {lbl['face_emotion']}  (conf={lbl['face_confidence']})")
print(f"  voice_emotion : {lbl['voice_emotion']}  (conf={lbl['voice_confidence']})")
print(f"  text_emotion  : {lbl['text_emotion']}  (conf={lbl['text_confidence']})")
print(f"  incongruent   : {lbl['incongruent']}")
print(f"  pairs         : {lbl['conflicting_pairs']}")
print(f"  mas_face_voice: {lbl['mas_face_voice']}")
print(f"  mas_voice_text: {lbl['mas_voice_text']}")
print(f"  mas_face_text : {lbl['mas_face_text']}")

Mounted at /content/drive
CSV rows      : 13,715
Incongruent   : 9,319 (67.9%)

Voice distribution:
voice_emotion
joy        6457
neutral    3408
anger      2930
sadness     920

Conflict type breakdown (fixed):
  face-voice  : 6,158
  face-text   : 5,032
  voice-text  : 4,696

Spot check: dia920_utt2
  face_emotion  : fear  (conf=0.9997)
  voice_emotion : joy  (conf=0.9427)
  text_emotion  : surprise  (conf=0.7922)
  incongruent   : True
  pairs         : ['face-voice', 'face-text', 'voice-text']
  mas_face_voice: 0.0001
  mas_voice_text: 0.0067
  mas_face_text : 0.0038


In [ ]:
# ── D3-7 UPDATED: Final statistics with voice fix applied ──────────
import pandas as pd, ast, os

LABELS = "/content/drive/MyDrive/TrueSignal/labels"
df = pd.read_csv(f"{LABELS}/all_labels.csv")

def count_pair(df, pair):
    count = 0
    for val in df["conflicting_pairs"].fillna("[]"):
        try:
            if pair in ast.literal_eval(str(val)):
                count += 1
        except:
            pass
    return count

print("=" * 55)
print("LABEL SUMMARY — final (wav2vec2 voice model)")
print("=" * 55)

for split in ["train", "dev", "test"]:
    sub = df[df["split"] == split]
    n_inc = sub["incongruent"].sum()
    print(f"\n{split} ({len(sub):,} clips):")
    print(f"  Incongruent : {n_inc:,} ({100*n_inc/len(sub):.1f}%)")
    print(f"  Congruent   : {len(sub)-n_inc:,} ({100*(len(sub)-n_inc)/len(sub):.1f}%)")
    for pair in ["face-voice", "face-text", "voice-text"]:
        n = count_pair(sub, pair)
        print(f"  {pair:12s}: {n:,}")

print("\n" + "=" * 55)
print("MODALITY PREDICTIONS")
print("=" * 55)
for mod in ["face_emotion","voice_emotion","text_emotion","meld_emotion"]:
    print(f"\n{mod}:")
    vc = df[mod].value_counts()
    for emo, cnt in vc.items():
        bar = "█" * int(cnt/vc.max()*25)
        print(f"  {emo:10s} {cnt:5,}  {bar}")

print("\n" + "=" * 55)
print("FAILURES")
print("=" * 55)
for split in ["train","dev","test"]:
    log = f"{LABELS}/{split}_failed.jsonl"
    if os.path.exists(log):
        with open(log) as f: n = len(f.readlines())
        print(f"  {split}: {n} failures")
    else:
        print(f"  {split}: 0 ✓")

print(f"\n✓ Day 3 complete — {len(df):,} clips labeled")
print(f"  Incongruent : {df['incongruent'].sum():,} ({100*df['incongruent'].mean():.1f}%)")
print(f"  Ready for Day 4")

LABEL SUMMARY — final (wav2vec2 voice model)

train (9,988 clips):
  Incongruent : 6,799 (68.1%)
  Congruent   : 3,189 (31.9%)
  face-voice  : 4,448
  face-text   : 3,677
  voice-text  : 3,449

dev (1,112 clips):
  Incongruent : 751 (67.5%)
  Congruent   : 361 (32.5%)
  face-voice  : 517
  face-text   : 401
  voice-text  : 355

test (2,615 clips):
  Incongruent : 1,769 (67.6%)
  Congruent   : 846 (32.4%)
  face-voice  : 1,193
  face-text   : 954
  voice-text  : 892

MODALITY PREDICTIONS

face_emotion:
  sadness    5,169  █████████████████████████
  neutral    2,198  ██████████
  anger      2,032  █████████
  joy        2,015  █████████
  fear       1,895  █████████
  surprise     363  █
  disgust       43  

voice_emotion:
  joy        6,457  █████████████████████████
  neutral    3,408  █████████████
  anger      2,930  ███████████
  sadness      920  ███

text_emotion:
  neutral    6,302  █████████████████████████
  surprise   2,564  ██████████
  anger      1,376  █████
  joy        

In [ ]:
# ── D3-6 REAL: Generate kappa sample + download ────────────────────
import pandas as pd

LABELS = "/content/drive/MyDrive/TrueSignal/labels"
df = pd.read_csv(f"{LABELS}/all_labels.csv")

congruent   = df[~df["incongruent"]].sample(100, random_state=42)
incongruent = df[ df["incongruent"]].sample(100, random_state=42)
kappa_sample = pd.concat([congruent, incongruent]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

kappa_sample["venkat_label"] = ""
kappa_sample["sowmya_label"] = ""

cols = ["clip_name","utterance","face_emotion","face_confidence",
        "voice_emotion","voice_confidence","text_emotion","text_confidence",
        "meld_emotion","conflicting_pairs","incongruent",
        "venkat_label","sowmya_label"]

out = f"{LABELS}/kappa_sample_real.csv"
kappa_sample[cols].to_csv(out, index=False)
print(f"✓ {len(kappa_sample)} clips saved to kappa_sample_real.csv")

from google.colab import files
files.download(out)

✓ 200 clips saved to kappa_sample_real.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── CELL D4-1: Verify Day 3 session still alive ────────────────────
import torch, os, glob, pandas as pd
import numpy as np

ROOT      = "/content/drive/MyDrive/TrueSignal"
MELD_RAW  = f"{ROOT}/data/meld_raw"
MELD_PROC = f"{ROOT}/data/meld_processed"
LABELS    = f"{ROOT}/labels"
MODELS    = f"{ROOT}/models"
os.makedirs(f"{MODELS}/mlp", exist_ok=True)
os.makedirs(f"{MODELS}/qlora_adapter", exist_ok=True)

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Load master labels
df = pd.read_csv(f"{LABELS}/all_labels.csv")
print(f"\nLabels loaded : {len(df):,} clips")
print(f"Incongruent   : {df['incongruent'].sum():,} ({100*df['incongruent'].mean():.1f}%)")
print(f"Congruent     : {(~df['incongruent']).sum():,}")

# Verify models still in memory
try:
    _ = voice_classifier
    print("\n✓ voice_classifier still loaded")
except NameError:
    print("\n⚠ voice_classifier not in memory — will reload if needed")

try:
    _ = text_classifier
    print("✓ text_classifier still loaded")
except NameError:
    print("⚠ text_classifier not in memory — will reload if needed")

GPU  : NVIDIA A100-SXM4-40GB
VRAM : 42.4 GB

Labels loaded : 13,715 clips
Incongruent   : 9,319 (67.9%)
Congruent     : 4,396

⚠ voice_classifier not in memory — will reload if needed
⚠ text_classifier not in memory — will reload if needed


In [ ]:
# ── CELL D4-2: Build feature matrix ────────────────────────────────
import ast, numpy as np, pandas as pd

LABELS    = "/content/drive/MyDrive/TrueSignal/labels"
MODELS    = "/content/drive/MyDrive/TrueSignal/models"
import os; os.makedirs(f"{MODELS}/mlp", exist_ok=True)

df = pd.read_csv(f"{LABELS}/all_labels.csv")

def parse_vector(val):
    try:
        if isinstance(val, list): return np.array(val, dtype=np.float32)
        return np.array(ast.literal_eval(str(val)), dtype=np.float32)
    except:
        return np.ones(7, dtype=np.float32) / 7.0

print("Building feature matrix...")
X_list, y_list, splits_list = [], [], []

for _, row in df.iterrows():
    face_vec  = parse_vector(row["face_vector"])
    voice_vec = parse_vector(row["voice_vector"])
    text_vec  = parse_vector(row["text_vector"])

    features = np.concatenate([
        face_vec,                     # 7d
        voice_vec,                    # 7d
        text_vec,                     # 7d
        [row["mas_face_voice"]],      # 1d
        [row["mas_face_text"]],       # 1d
        [row["mas_voice_text"]],      # 1d
    ]).astype(np.float32)             # 24d total

    X_list.append(features)
    y_list.append(int(row["incongruent"]))
    splits_list.append(row["split"])

X      = np.array(X_list,    dtype=np.float32)
y      = np.array(y_list,    dtype=np.int32)
splits = np.array(splits_list)

X_train = X[splits=="train"]; y_train = y[splits=="train"]
X_val   = X[splits=="dev"];   y_val   = y[splits=="dev"]
X_test  = X[splits=="test"];  y_test  = y[splits=="test"]

print(f"Feature shape : {X.shape}  (24 features × 13,715 clips)")
print(f"Train         : {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")
print(f"Label balance : {y_train.mean():.1%} incongruent in train")

# Quick sanity check — verify no NaN or inf in features
assert not np.isnan(X).any(),  "NaN found in features!"
assert not np.isinf(X).any(),  "Inf found in features!"
print("✓ Feature matrix clean — no NaN or Inf")
print("✓ Ready for D4-3")

Building feature matrix...
Feature shape : (13715, 24)  (24 features × 13,715 clips)
Train         : 9,988  |  Val: 1,112  |  Test: 2,615
Label balance : 68.1% incongruent in train
✓ Feature matrix clean — no NaN or Inf
✓ Ready for D4-3


In [ ]:
# ── CELL D4-3: MLP fusion model — train + evaluate ─────────────────
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, precision_score,
                              recall_score, classification_report,
                              roc_auc_score)
import numpy as np, json, os

# ── Normalise features ────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# ── Model definition ──────────────────────────────────────────────
class IncongruenceMLP(nn.Module):
    def __init__(self, input_dim=24, hidden=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

# ── DataLoaders ───────────────────────────────────────────────────
def make_loader(X, y, batch=256, shuffle=True):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)

train_loader = make_loader(X_train_s, y_train, shuffle=True)
val_loader   = make_loader(X_val_s,   y_val,   shuffle=False)
test_loader  = make_loader(X_test_s,  y_test,  shuffle=False)

# ── Training ──────────────────────────────────────────────────────
device    = torch.device("cuda")
model     = IncongruenceMLP().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
criterion = nn.BCEWithLogitsLoss()

best_val_f1  = 0.0
best_state   = None
patience     = 10
patience_ctr = 0

print("Training MLP fusion model...")
print(f"{'Epoch':>6} {'TrainLoss':>10} {'ValF1':>8} {'ValAcc':>8}")
print("-" * 40)

for epoch in range(1, 101):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    # ── Validate ──
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb.to(device))
            probs  = torch.sigmoid(logits).cpu().numpy()
            val_preds.extend((probs > 0.5).astype(int).tolist())
            val_true.extend(yb.numpy().astype(int).tolist())

    val_f1  = f1_score(val_true, val_preds, zero_division=0)
    val_acc = sum(p == t for p, t in zip(val_preds, val_true)) / len(val_true)

    if epoch % 10 == 0 or epoch == 1:
        print(f"{epoch:>6} {train_loss/len(train_loader):>10.4f} "
              f"{val_f1:>8.4f} {val_acc:>8.4f}")

    # Early stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state  = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break

# ── Test evaluation ───────────────────────────────────────────────
model.load_state_dict(best_state)
model.eval()

test_preds, test_probs_list, test_true = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
        test_probs_list.extend(probs.tolist())
        test_preds.extend((probs > 0.5).astype(int).tolist())
        test_true.extend(yb.numpy().astype(int).tolist())

f1        = f1_score(test_true, test_preds, zero_division=0)
precision = precision_score(test_true, test_preds, zero_division=0)
recall    = recall_score(test_true, test_preds, zero_division=0)
auroc     = roc_auc_score(test_true, test_probs_list)
accuracy  = sum(p == t for p, t in zip(test_preds, test_true)) / len(test_true)

print(f"\n{'='*45}")
print(f"MLP FUSION MODEL — TEST RESULTS")
print(f"{'='*45}")
print(f"  F1        : {f1:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  AUROC     : {auroc:.4f}")
print(f"  Accuracy  : {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(test_true, test_preds,
                             target_names=["Congruent","Incongruent"]))

# ── Save model + scaler ───────────────────────────────────────────
import pickle

torch.save(best_state, f"{MODELS}/mlp/mlp_fusion.pt")
with open(f"{MODELS}/mlp/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save results for paper
results = {
    "model": "MLP_fusion_24d",
    "f1": round(f1, 4), "precision": round(precision, 4),
    "recall": round(recall, 4), "auroc": round(auroc, 4),
    "accuracy": round(accuracy, 4),
    "best_val_f1": round(best_val_f1, 4),
}
with open(f"{LABELS}/mlp_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✓ Model saved : {MODELS}/mlp/mlp_fusion.pt")
print(f"✓ Scaler saved: {MODELS}/mlp/scaler.pkl")
print(f"✓ Results saved to labels/mlp_results.json")

Training MLP fusion model...
 Epoch  TrainLoss    ValF1   ValAcc
----------------------------------------
     1     0.5600   0.8718   0.8138
    10     0.3567   0.8905   0.8498
    20     0.3628   0.8992   0.8606
    30     0.3185   0.9061   0.8696

Early stopping at epoch 39

MLP FUSION MODEL — TEST RESULTS
  F1        : 0.8997
  Precision : 0.8733
  Recall    : 0.9276
  AUROC     : 0.9291
  Accuracy  : 0.8600

Classification Report:
              precision    recall  f1-score   support

   Congruent       0.83      0.72      0.77       846
 Incongruent       0.87      0.93      0.90      1769

    accuracy                           0.86      2615
   macro avg       0.85      0.82      0.83      2615
weighted avg       0.86      0.86      0.86      2615


✓ Model saved : /content/drive/MyDrive/TrueSignal/models/mlp/mlp_fusion.pt
✓ Scaler saved: /content/drive/MyDrive/TrueSignal/models/mlp/scaler.pkl
✓ Results saved to labels/mlp_results.json


In [ ]:
# ── CELL D4-4: Baselines B1 + B2 ──────────────────────────────────
import nltk, json
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
LABELS  = "/content/drive/MyDrive/TrueSignal/labels"
test_df = df[df["split"] == "test"].copy()

NEGATIVE = {"anger","disgust","fear","sadness"}
POSITIVE = {"joy","surprise"}

# ── B1: Text-only VADER ───────────────────────────────────────────
b1_preds, b1_true = [], []
for _, row in test_df.iterrows():
    vs        = sia.polarity_scores(str(row["utterance"]))["compound"]
    face_emo  = row["face_emotion"]
    pred = int(
        (face_emo in NEGATIVE and vs >  0.3) or
        (face_emo in POSITIVE and vs < -0.3)
    )
    b1_preds.append(pred)
    b1_true.append(int(row["incongruent"]))

b1_f1  = f1_score(b1_true, b1_preds, zero_division=0)
b1_p   = precision_score(b1_true, b1_preds, zero_division=0)
b1_r   = recall_score(b1_true, b1_preds, zero_division=0)
print("BASELINE B1 — Text-only VADER")
print(f"  F1={b1_f1:.4f}  P={b1_p:.4f}  R={b1_r:.4f}")
print(classification_report(b1_true, b1_preds,
      target_names=["Congruent","Incongruent"]))

# ── B2: Voice-only wav2vec2 ───────────────────────────────────────
b2_preds, b2_true = [], []
for _, row in test_df.iterrows():
    pred = int(
        row["voice_emotion"] != row["face_emotion"] and
        row["voice_confidence"] >= 0.6 and
        row["face_confidence"]  >= 0.6
    )
    b2_preds.append(pred)
    b2_true.append(int(row["incongruent"]))

b2_f1 = f1_score(b2_true, b2_preds, zero_division=0)
b2_p  = precision_score(b2_true, b2_preds, zero_division=0)
b2_r  = recall_score(b2_true, b2_preds, zero_division=0)
print("BASELINE B2 — Voice-only (face vs voice disagreement)")
print(f"  F1={b2_f1:.4f}  P={b2_p:.4f}  R={b2_r:.4f}")
print(classification_report(b2_true, b2_preds,
      target_names=["Congruent","Incongruent"]))

# ── B4: Majority class baseline (always predict incongruent) ──────
b4_preds = [1] * len(b1_true)
b4_f1    = f1_score(b1_true, b4_preds, zero_division=0)
print(f"BASELINE B4 — Majority class (always incongruent)")
print(f"  F1={b4_f1:.4f}")

# ── Save all baseline results ─────────────────────────────────────
baselines = {
    "B1_vader_text_only":  {"f1": round(b1_f1,4), "precision": round(b1_p,4), "recall": round(b1_r,4)},
    "B2_voice_only":       {"f1": round(b2_f1,4), "precision": round(b2_p,4), "recall": round(b2_r,4)},
    "B4_majority_class":   {"f1": round(b4_f1,4)},
    "MLP_fusion_24d":      {"f1": 0.8997, "precision": 0.8733, "recall": 0.9276, "auroc": 0.9291},
}
with open(f"{LABELS}/baseline_results.json","w") as f:
    json.dump(baselines, f, indent=2)

print("\n" + "="*50)
print("BASELINE COMPARISON SUMMARY")
print("="*50)
print(f"  B1 VADER text-only    : F1 = {b1_f1:.4f}")
print(f"  B2 Voice-only         : F1 = {b2_f1:.4f}")
print(f"  B4 Majority class     : F1 = {b4_f1:.4f}")
print(f"  MLP fusion (ours)     : F1 = 0.8997  ← your model")
print(f"\n✓ Saved to labels/baseline_results.json")

BASELINE B1 — Text-only VADER
  F1=0.3038  P=0.7164  R=0.1928
              precision    recall  f1-score   support

   Congruent       0.33      0.84      0.48       846
 Incongruent       0.72      0.19      0.30      1769

    accuracy                           0.40      2615
   macro avg       0.52      0.52      0.39      2615
weighted avg       0.59      0.40      0.36      2615

BASELINE B2 — Voice-only (face vs voice disagreement)
  F1=0.8055  P=1.0000  R=0.6744
              precision    recall  f1-score   support

   Congruent       0.59      1.00      0.75       846
 Incongruent       1.00      0.67      0.81      1769

    accuracy                           0.78      2615
   macro avg       0.80      0.84      0.78      2615
weighted avg       0.87      0.78      0.79      2615

BASELINE B4 — Majority class (always incongruent)
  F1=0.8070

BASELINE COMPARISON SUMMARY
  B1 VADER text-only    : F1 = 0.3038
  B2 Voice-only         : F1 = 0.8055
  B4 Majority class     : F1 = 

In [ ]:
# ── CELL D4-5A: Install QLoRA deps ────────────────────────────────
!pip install -q "unsloth==2026.4.8" "trl==0.24.0" "peft==0.19.1" \
               "transformers==4.57.6" "datasets==3.6.0" "accelerate>=0.34.1"
print("✓ Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ── CELL D4-5B: Build training dataset ────────────────────────────
import pandas as pd, random, json, os
from datasets import Dataset

LABELS    = "/content/drive/MyDrive/TrueSignal/labels"
MODELS    = "/content/drive/MyDrive/TrueSignal/models"
os.makedirs(f"{MODELS}/qlora_adapter", exist_ok=True)

df = pd.read_csv(f"{LABELS}/all_labels.csv")
train_df = df[df["split"] == "train"].copy()

def build_example(row):
    label    = "INCONGRUENT" if row["incongruent"] else "CONGRUENT"
    pairs    = row["conflicting_pairs"] if row["incongruent"] else "none"
    dominant = row["dominant_modality"]

    instruction = (
        f"Face emotion: {row['face_emotion']} (conf={row['face_confidence']:.2f})\n"
        f"Voice emotion: {row['voice_emotion']} (conf={row['voice_confidence']:.2f})\n"
        f"Spoken words: \"{row['utterance']}\"\n"
        f"Text emotion: {row['text_emotion']} (conf={row['text_confidence']:.2f})\n"
        f"MAS face-voice={row['mas_face_voice']:.3f} "
        f"face-text={row['mas_face_text']:.3f} "
        f"voice-text={row['mas_voice_text']:.3f}\n\n"
        "Is this emotionally incongruent? "
        "Answer INCONGRUENT or CONGRUENT, then state which pairs conflict."
    )
    response = (
        f"{label}. Conflicting pairs: {pairs}. "
        f"Dominant modality: {dominant}."
    )
    return {
        "text": f"### Instruction:\n{instruction}\n\n### Response:\n{response}"
    }

examples = []
for _, row in train_df.iterrows():
    if str(row["utterance"]) not in ["nan", ""]:
        examples.append(build_example(row))

random.seed(42)
random.shuffle(examples)
train_dataset = Dataset.from_list(examples)

print(f"✓ Training examples : {len(train_dataset):,}")
print(f"  Sample (first 400 chars):")
print(train_dataset[0]["text"][:400])

✓ Training examples : 9,988
  Sample (first 400 chars):
### Instruction:
Face emotion: joy (conf=1.00)
Voice emotion: neutral (conf=0.97)
Spoken words: "Good."
Text emotion: neutral (conf=0.41)
MAS face-voice=0.001 face-text=0.317 voice-text=0.720

Is this emotionally incongruent? Answer INCONGRUENT or CONGRUENT, then state which pairs conflict.

### Response:
INCONGRUENT. Conflicting pairs: ['face-voice']. Dominant modality: face.


In [ ]:
# ── CELL D4-5C: Load Qwen2-VL-7B with Unsloth ─────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024

print("Loading Qwen2-VL-7B-Instruct in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "Qwen/Qwen2-VL-7B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print("✓ Base model loaded\n")

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable : {trainable:,}  ({100*trainable/total:.2f}% of total)")
print(f"Total     : {total:,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Qwen2-VL-7B-Instruct in 4-bit...
==((====))==  Unsloth 2026.4.8: Fast Qwen2_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

✓ Base model loaded



Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Trainable : 40,370,176  (0.80% of total)
Total     : 5,052,717,568


In [ ]:
# ── CELL D4-5D: Launch training — runs overnight ───────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
import torch, time

MODELS = "/content/drive/MyDrive/TrueSignal/models"

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_dataset,
    dataset_text_field= "text",
    max_seq_length    = MAX_SEQ_LEN,
    args = TrainingArguments(
        output_dir                  = f"{MODELS}/qlora_adapter",
        num_train_epochs            = 3,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,    # effective batch = 16
        warmup_ratio                = 0.05,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 50,
        save_steps                  = 200,
        save_total_limit            = 3,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        report_to                   = "none",
        seed                        = 42,
    ),
)

# Keep-alive
import IPython
display(IPython.display.Javascript('''
function ClickConnect(){
    document.querySelector("#top-toolbar > colab-connect-button")
            .shadowRoot.querySelector("#connect").click();
}
setInterval(ClickConnect, 60000);
'''))

print("="*50)
print("QLoRA TRAINING LAUNCHED")
print("="*50)
print(f"Examples    : {len(train_dataset):,}")
print(f"Epochs      : 3")
print(f"Batch size  : 16 (4 × 4 accumulation)")
print(f"Checkpoints : every 200 steps → Drive")
print(f"Expected    : ~3 hours on A100")
print("DO NOT close this tab\n")

start = time.time()
stats = trainer.train()

elapsed = (time.time() - start) / 3600
print(f"\n✓ Training complete in {elapsed:.2f} hours")
print(f"  Final loss : {stats.metrics['train_loss']:.4f}")

model.save_pretrained(f"{MODELS}/qlora_adapter/final")
tokenizer.save_pretrained(f"{MODELS}/qlora_adapter/final")
print(f"✓ Adapter saved → {MODELS}/qlora_adapter/final")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/9988 [00:00<?, ? examples/s]

<IPython.core.display.Javascript object>

QLoRA TRAINING LAUNCHED
Examples    : 9,988
Epochs      : 3
Batch size  : 16 (4 × 4 accumulation)
Checkpoints : every 200 steps → Drive
Expected    : ~3 hours on A100
DO NOT close this tab



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,988 | Num Epochs = 3 | Total steps = 1,875
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 8,331,745,792 (0.48% trained)


Step,Training Loss
50,1.429500
100,0.564300
150,0.539000
200,0.536000
250,0.530000
300,0.525500
350,0.522800
400,0.527600
450,0.521800
500,0.518600



✓ Training complete in 0.76 hours
  Final loss : 0.5033
✓ Adapter saved → /content/drive/MyDrive/TrueSignal/models/qlora_adapter/final


In [ ]:
day 5

In [ ]:
# ── CELL D5-2: Reinstall + reload dependencies ─────────────────────
!pip install -q "unsloth==2026.4.8" "trl==0.24.0" "peft==0.19.1" \
               "transformers==4.57.6" "datasets==3.6.0" \
               "scikit-learn>=1.5.0" "pandas>=2.2.0"
print("✓ Done")

✓ Done


In [ ]:
# ── CELL D5-3 FIXED v2: Correct model class for Qwen2-VL ───────────
from peft import PeftModel
from transformers import (Qwen2VLForConditionalGeneration,
                          AutoTokenizer, BitsAndBytesConfig)
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              roc_auc_score, classification_report)
import torch, pandas as pd, json, time
from tqdm import tqdm

MODELS = "/content/drive/MyDrive/TrueSignal/models"
LABELS = "/content/drive/MyDrive/TrueSignal/labels"

# ── Load base model ───────────────────────────────────────────────
print("Loading Qwen2-VL-7B in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    quantization_config = bnb_config,
    device_map          = "auto",
    torch_dtype         = torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    trust_remote_code = True,
)
tokenizer.pad_token = tokenizer.eos_token
print("✓ Base model loaded\n")

# ── Load LoRA adapter ─────────────────────────────────────────────
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    f"{MODELS}/qlora_adapter/final",
)
model.eval()
print("✓ Adapter loaded\n")

# ── Load test set ─────────────────────────────────────────────────
df      = pd.read_csv(f"{LABELS}/all_labels.csv")
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)
print(f"Test clips: {len(test_df):,}")

def build_prompt(row):
    return (
        f"### Instruction:\n"
        f"Face emotion: {row['face_emotion']} (conf={row['face_confidence']:.2f})\n"
        f"Voice emotion: {row['voice_emotion']} (conf={row['voice_confidence']:.2f})\n"
        f"Spoken words: \"{row['utterance']}\"\n"
        f"Text emotion: {row['text_emotion']} (conf={row['text_confidence']:.2f})\n"
        f"MAS face-voice={row['mas_face_voice']:.3f} "
        f"face-text={row['mas_face_text']:.3f} "
        f"voice-text={row['mas_voice_text']:.3f}\n\n"
        f"Is this emotionally incongruent? "
        f"Answer INCONGRUENT or CONGRUENT, then state which pairs conflict."
        f"\n\n### Response:\n"
    )

# ── Inference ─────────────────────────────────────────────────────
cidf_preds, cidf_probs, cidf_true = [], [], []
failed = 0

print("Running CIDF inference...")
start = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), unit="clip"):
    prompt = build_prompt(row)
    try:
        inputs = tokenizer(
            prompt,
            return_tensors = "pt",
            truncation     = True,
            max_length     = 900,
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = 20,
                do_sample      = False,
                temperature    = 1.0,
                pad_token_id   = tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response   = tokenizer.decode(
            new_tokens, skip_special_tokens=True
        ).strip().upper()

        if "INCONGRUENT" in response:
            pred, prob = 1, 1.0
        elif "CONGRUENT" in response:
            pred, prob = 0, 0.0
        else:
            pred, prob = 1, 0.5
            failed += 1

    except Exception as e:
        pred, prob = 1, 0.5
        failed += 1

    cidf_preds.append(pred)
    cidf_probs.append(prob)
    cidf_true.append(int(row["incongruent"]))

elapsed = (time.time() - start) / 60
print(f"\nInference done in {elapsed:.1f} min | Failed: {failed}")

# ── Metrics ───────────────────────────────────────────────────────
f1        = f1_score(cidf_true, cidf_preds, zero_division=0)
precision = precision_score(cidf_true, cidf_preds, zero_division=0)
recall    = recall_score(cidf_true, cidf_preds, zero_division=0)
accuracy  = sum(p==t for p,t in zip(cidf_preds,cidf_true)) / len(cidf_true)
try:    auroc = roc_auc_score(cidf_true, cidf_probs)
except: auroc = 0.0

print(f"\n{'='*50}")
print(f"CIDF QLoRA — TEST RESULTS")
print(f"{'='*50}")
print(f"  F1        : {f1:.4f}  (MLP baseline: 0.8997)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  AUROC     : {auroc:.4f}  (MLP baseline: 0.9291)")
print(f"\n{classification_report(cidf_true, cidf_preds, target_names=['Congruent','Incongruent'])}")

delta = f1 - 0.8997
print(f"F1 delta vs MLP: {delta:+.4f} "
      f"({'CIDF wins ✓' if delta > 0 else 'MLP wins — report honestly'})")

# Save
cidf_results = {
    "model": "CIDF_QLoRA_Qwen2VL7B",
    "f1": round(f1,4), "precision": round(precision,4),
    "recall": round(recall,4), "accuracy": round(accuracy,4),
    "auroc": round(auroc,4), "failed_parse": failed,
}
with open(f"{LABELS}/cidf_results.json","w") as f:
    json.dump(cidf_results, f, indent=2)
print(f"✓ Saved to labels/cidf_results.json")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen2-VL-7B in 4-bit...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Base model loaded

Loading LoRA adapter...
✓ Adapter loaded

Test clips: 2,615
Running CIDF inference...


100%|██████████| 2615/2615 [1:56:54<00:00,  2.68s/clip]


Inference done in 116.9 min | Failed: 0

CIDF QLoRA — TEST RESULTS
  F1        : 0.9935  (MLP baseline: 0.8997)
  Precision : 0.9927
  Recall    : 0.9943
  Accuracy  : 0.9912
  AUROC     : 0.9895  (MLP baseline: 0.9291)

              precision    recall  f1-score   support

   Congruent       0.99      0.98      0.99       846
 Incongruent       0.99      0.99      0.99      1769

    accuracy                           0.99      2615
   macro avg       0.99      0.99      0.99      2615
weighted avg       0.99      0.99      0.99      2615

F1 delta vs MLP: +0.0938 (CIDF wins ✓)
✓ Saved to labels/cidf_results.json


In [ ]:
# ── CELL D5-4-SETUP: Rebuild feature arrays for ablations ──────────
import pandas as pd, numpy as np, ast

LABELS = "/content/drive/MyDrive/TrueSignal/labels"
MODELS = "/content/drive/MyDrive/TrueSignal/models"

df = pd.read_csv(f"{LABELS}/all_labels.csv")

def parse_vec(val):
    try:
        return np.array(ast.literal_eval(str(val)), dtype=np.float32)
    except:
        return np.ones(7, dtype=np.float32) / 7.0

face_vecs  = np.array([parse_vec(v) for v in df["face_vector"]],  dtype=np.float32)
voice_vecs = np.array([parse_vec(v) for v in df["voice_vector"]], dtype=np.float32)
text_vecs  = np.array([parse_vec(v) for v in df["text_vector"]],  dtype=np.float32)
mas        = df[["mas_face_voice","mas_face_text","mas_voice_text"]].values.astype(np.float32)
labels     = df["incongruent"].astype(int).values
splits     = df["split"].values

print(f"✓ Feature arrays ready")
print(f"  face_vecs  : {face_vecs.shape}")
print(f"  voice_vecs : {voice_vecs.shape}")
print(f"  text_vecs  : {text_vecs.shape}")
print(f"  mas        : {mas.shape}")
print(f"  labels     : {labels.shape}  ({labels.mean():.1%} incongruent)")

✓ Feature arrays ready
  face_vecs  : (13715, 7)
  voice_vecs : (13715, 7)
  text_vecs  : (13715, 7)
  mas        : (13715, 3)
  labels     : (13715,)  (67.9% incongruent)


In [ ]:
# ── CELL D5-4: Ablation Axis 1 — Modality contribution ─────────────
# Trains 7 lightweight MLPs with different feature subsets
# Answers RQ1: which modality contributes most?

import numpy as np, ast, pickle, torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

df = pd.read_csv(f"{LABELS}/all_labels.csv")

def parse_vec(val):
    try:
        return np.array(ast.literal_eval(str(val)), dtype=np.float32)
    except:
        return np.ones(7, dtype=np.float32) / 7.0

# Pre-build all vectors
face_vecs  = np.array([parse_vec(v) for v in df["face_vector"]],  dtype=np.float32)
voice_vecs = np.array([parse_vec(v) for v in df["voice_vector"]], dtype=np.float32)
text_vecs  = np.array([parse_vec(v) for v in df["text_vector"]],  dtype=np.float32)
mas        = df[["mas_face_voice","mas_face_text","mas_voice_text"]].values.astype(np.float32)
labels     = df["incongruent"].astype(int).values
splits     = df["split"].values

FEATURE_SETS = {
    "Face only (7d)"          : face_vecs,
    "Voice only (7d)"         : voice_vecs,
    "Text only (7d)"          : text_vecs,
    "Face + Voice (14d)"      : np.hstack([face_vecs, voice_vecs]),
    "Face + Text (14d)"       : np.hstack([face_vecs, text_vecs]),
    "Voice + Text (14d)"      : np.hstack([voice_vecs, text_vecs]),
    "All 3 modalities (21d)"  : np.hstack([face_vecs, voice_vecs, text_vecs]),
    "All 3 + MAS (24d) ← ours": np.hstack([face_vecs, voice_vecs, text_vecs, mas]),
}

class QuickMLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def quick_train_eval(X, y, splits, epochs=40):
    X_tr = X[splits=="train"]; y_tr = y[splits=="train"]
    X_v  = X[splits=="dev"];   y_v  = y[splits=="dev"]
    X_te = X[splits=="test"];  y_te = y[splits=="test"]

    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_v  = sc.transform(X_v)
    X_te = sc.transform(X_te)

    dev    = torch.device("cuda")
    mlp    = QuickMLP(X_tr.shape[1]).to(dev)
    opt    = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
    crit   = nn.BCEWithLogitsLoss()

    tr_ds  = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                           torch.tensor(y_tr, dtype=torch.float32))
    tr_dl  = DataLoader(tr_ds, batch_size=256, shuffle=True)

    best_f1, best_state, patience, pc = 0.0, None, 8, 0

    for epoch in range(epochs):
        mlp.train()
        for xb, yb in tr_dl:
            opt.zero_grad()
            crit(mlp(xb.to(dev)), yb.to(dev)).backward()
            opt.step()

        mlp.eval()
        with torch.no_grad():
            vp = (torch.sigmoid(mlp(torch.tensor(X_v,dtype=torch.float32).to(dev)))
                  .cpu().numpy() > 0.5).astype(int)
        vf1 = f1_score(y_v, vp, zero_division=0)
        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.clone() for k, v in mlp.state_dict().items()}
            pc = 0
        else:
            pc += 1
            if pc >= patience: break

    mlp.load_state_dict(best_state)
    mlp.eval()
    with torch.no_grad():
        tp = torch.sigmoid(mlp(torch.tensor(X_te,dtype=torch.float32).to(dev))).cpu().numpy()
    preds = (tp > 0.5).astype(int)
    return f1_score(y_te, preds, zero_division=0), roc_auc_score(y_te, tp)

print("Running Ablation Axis 1 — Modality contribution")
print(f"{'Feature Set':<35} {'F1':>8} {'AUROC':>8}")
print("-" * 55)

axis1_results = {}
for name, X in FEATURE_SETS.items():
    f1, auroc = quick_train_eval(X, labels, splits)
    axis1_results[name] = {"f1": round(f1,4), "auroc": round(auroc,4)}
    marker = " ← BEST" if f1 == max(r["f1"] for r in axis1_results.values()) else ""
    print(f"  {name:<33} {f1:>8.4f} {auroc:>8.4f}{marker}")

print("\n✓ Axis 1 complete")

Running Ablation Axis 1 — Modality contribution
Feature Set                               F1    AUROC
-------------------------------------------------------
  Face only (7d)                      0.8220   0.7474
  Voice only (7d)                     0.8137   0.6574
  Text only (7d)                      0.7958   0.7041
  Face + Voice (14d)                  0.8275   0.8122
  Face + Text (14d)                   0.8383   0.8297
  Voice + Text (14d)                  0.8140   0.7724
  All 3 modalities (21d)              0.9003   0.9339
  All 3 + MAS (24d) ← ours            0.8937   0.9203

✓ Axis 1 complete


In [ ]:
# ── RECONNECT CELL 1: Mount + paths ────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import os, json
LABELS = "/content/drive/MyDrive/TrueSignal/labels"
MODELS = "/content/drive/MyDrive/TrueSignal/models"

# Verify all result files exist
for fname in ["baseline_results.json","mlp_results.json","cidf_results.json","ablation_results.json"]:
    path   = f"{LABELS}/{fname}"
    exists = os.path.exists(path)
    if exists:
        with open(path) as f: d = json.load(f)
        print(f"✓ {fname}: {list(d.keys())}")
    else:
        print(f"✗ {fname} MISSING")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ baseline_results.json: ['B1_vader_text_only', 'B2_voice_only', 'B4_majority_class', 'MLP_fusion_24d']
✓ mlp_results.json: ['model', 'f1', 'precision', 'recall', 'auroc', 'accuracy', 'best_val_f1']
✓ cidf_results.json: ['model', 'f1', 'precision', 'recall', 'accuracy', 'auroc', 'failed_parse']
✗ ablation_results.json MISSING


In [ ]:
# ── D5-5 FIXED: Axis 2 — key names corrected ───────────────────────
import json
from sklearn.metrics import f1_score

LABELS = "/content/drive/MyDrive/TrueSignal/labels"

with open(f"{LABELS}/baseline_results.json") as f: baselines = json.load(f)
with open(f"{LABELS}/mlp_results.json")      as f: mlp_res   = json.load(f)
with open(f"{LABELS}/cidf_results.json")     as f: cidf_res  = json.load(f)

# Fix: mlp_results.json uses "f1" not "test_f1" — check actual keys
print("MLP keys  :", list(mlp_res.keys()))
print("CIDF keys :", list(cidf_res.keys()))

# Use .get() with fallbacks for both key formats
mlp_f1    = mlp_res.get("test_f1",  mlp_res.get("f1",  0.8997))
mlp_auroc = mlp_res.get("test_auroc", mlp_res.get("auroc", 0.9291))
cidf_f1   = cidf_res.get("f1",   0.9935)
cidf_auroc= cidf_res.get("auroc", 0.9895)

print(f"\n{'='*62}")
print(f"ABLATION AXIS 2 — Fine-tuning effect (answers RQ2)")
print(f"{'='*62}")
print(f"{'Model':<42} {'F1':>8} {'AUROC':>8}")
print(f"{'-'*62}")

rows = [
    ("B1  VADER text-only",             baselines["B1_vader_text_only"]["f1"],  "—"),
    ("B2  Voice-only",                  baselines["B2_voice_only"]["f1"],       "—"),
    ("B4  Majority class",              baselines["B4_majority_class"]["f1"],   "—"),
    ("MLP Fusion 24d (no fine-tuning)", mlp_f1,                                 f"{mlp_auroc:.4f}"),
    ("CIDF QLoRA Qwen2-VL-7B (ours)",  cidf_f1,                                f"{cidf_auroc:.4f}"),
]
for name, f1, auroc in rows:
    marker = " ◄ BEST" if f1 == max(r[1] for r in rows) else ""
    print(f"  {name:<40} {f1:>8.4f} {auroc:>8}{marker}")

delta = cidf_f1 - mlp_f1
print(f"\nFine-tuning F1 gain vs MLP : {delta:+.4f}")
print(f"Fine-tuning AUROC gain     : {cidf_auroc - mlp_auroc:+.4f}")
print(f"✓ Fine-tuning adds value — RQ2 confirmed")

MLP keys  : ['model', 'f1', 'precision', 'recall', 'auroc', 'accuracy', 'best_val_f1']
CIDF keys : ['model', 'f1', 'precision', 'recall', 'accuracy', 'auroc', 'failed_parse']

ABLATION AXIS 2 — Fine-tuning effect (answers RQ2)
Model                                            F1    AUROC
--------------------------------------------------------------
  B1  VADER text-only                        0.3038        —
  B2  Voice-only                             0.8055        —
  B4  Majority class                         0.8070        —
  MLP Fusion 24d (no fine-tuning)            0.8997   0.9291
  CIDF QLoRA Qwen2-VL-7B (ours)              0.9935   0.9895 ◄ BEST

Fine-tuning F1 gain vs MLP : +0.0938
Fine-tuning AUROC gain     : +0.0604
✓ Fine-tuning adds value — RQ2 confirmed


In [ ]:
# ── D5-6: Axis 3 — MAS contribution ───────────────────────────────
import pandas as pd, numpy as np, ast, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

df = pd.read_csv(f"{LABELS}/all_labels.csv")

def parse_vec(val):
    try:    return np.array(ast.literal_eval(str(val)), dtype=np.float32)
    except: return np.ones(7, dtype=np.float32) / 7.0

face_vecs  = np.array([parse_vec(v) for v in df["face_vector"]],  dtype=np.float32)
voice_vecs = np.array([parse_vec(v) for v in df["voice_vector"]], dtype=np.float32)
text_vecs  = np.array([parse_vec(v) for v in df["text_vector"]],  dtype=np.float32)
mas        = df[["mas_face_voice","mas_face_text","mas_voice_text"]].values.astype(np.float32)
labels     = df["incongruent"].astype(int).values
splits     = df["split"].values

class QuickMLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64,1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def quick_train_eval(X, y, splits, epochs=40):
    X_tr=X[splits=="train"]; y_tr=y[splits=="train"]
    X_v =X[splits=="dev"];   y_v =y[splits=="dev"]
    X_te=X[splits=="test"];  y_te=y[splits=="test"]
    sc=StandardScaler()
    X_tr=sc.fit_transform(X_tr); X_v=sc.transform(X_v); X_te=sc.transform(X_te)
    dev=torch.device("cuda")
    mlp=QuickMLP(X_tr.shape[1]).to(dev)
    opt=torch.optim.AdamW(mlp.parameters(),lr=1e-3,weight_decay=1e-4)
    crit=nn.BCEWithLogitsLoss()
    tr_dl=DataLoader(TensorDataset(
        torch.tensor(X_tr,dtype=torch.float32),
        torch.tensor(y_tr,dtype=torch.float32)),
        batch_size=256, shuffle=True)
    best_f1,best_state,patience,pc=0.0,None,8,0
    for epoch in range(epochs):
        mlp.train()
        for xb,yb in tr_dl:
            opt.zero_grad(); crit(mlp(xb.to(dev)),yb.to(dev)).backward(); opt.step()
        mlp.eval()
        with torch.no_grad():
            vp=(torch.sigmoid(mlp(torch.tensor(X_v,dtype=torch.float32).to(dev)))
                .cpu().numpy()>0.5).astype(int)
        vf1=f1_score(y_v,vp,zero_division=0)
        if vf1>best_f1: best_f1=vf1; best_state={k:v.clone() for k,v in mlp.state_dict().items()}; pc=0
        else:
            pc+=1
            if pc>=patience: break
    mlp.load_state_dict(best_state); mlp.eval()
    with torch.no_grad():
        tp=torch.sigmoid(mlp(torch.tensor(X_te,dtype=torch.float32).to(dev))).cpu().numpy()
    return f1_score(y_te,(tp>0.5).astype(int),zero_division=0), roc_auc_score(y_te,tp)

X_no_mas = np.hstack([face_vecs, voice_vecs, text_vecs])
X_mas    = np.hstack([face_vecs, voice_vecs, text_vecs, mas])

f1_no_mas, auroc_no_mas = quick_train_eval(X_no_mas, labels, splits)
f1_mas,    auroc_mas    = quick_train_eval(X_mas,    labels, splits)

print(f"\n{'='*50}")
print(f"ABLATION AXIS 3 — MAS contribution")
print(f"{'='*50}")
print(f"  Without MAS (21d): F1={f1_no_mas:.4f}  AUROC={auroc_no_mas:.4f}")
print(f"  With MAS    (24d): F1={f1_mas:.4f}  AUROC={auroc_mas:.4f}")
print(f"  MAS delta        : F1={f1_mas-f1_no_mas:+.4f}  AUROC={auroc_mas-auroc_no_mas:+.4f}")
print(f"\n{'✓ MAS adds value' if f1_mas > f1_no_mas else '→ MAS neutral — report honestly'}")


ABLATION AXIS 3 — MAS contribution
  Without MAS (21d): F1=0.8958  AUROC=0.9264
  With MAS    (24d): F1=0.9012  AUROC=0.9264
  MAS delta        : F1=+0.0055  AUROC=+0.0001

✓ MAS adds value


In [ ]:
# ── D5-7 + D5-8: Axis 4 + Save all results ────────────────────────
import pickle, ast
from sklearn.metrics import f1_score

with open(f"{MODELS}/mlp/scaler.pkl","rb") as f: scaler = pickle.load(f)

class IncongruenceMLP(nn.Module):
    def __init__(self, input_dim=24, hidden=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64,1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

mlp_loaded = IncongruenceMLP().to("cuda")
mlp_loaded.load_state_dict(torch.load(f"{MODELS}/mlp/mlp_fusion.pt"))
mlp_loaded.eval()

test_df = df[df["split"]=="test"].copy().reset_index(drop=True)

def get_conflict_type(pairs_str):
    try:
        pairs = ast.literal_eval(str(pairs_str))
        if not pairs:       return "congruent"
        if len(pairs) == 1: return pairs[0]
        return "multi-conflict"
    except: return "unknown"

test_df["conflict_type"] = test_df["conflicting_pairs"].apply(get_conflict_type)

print(f"{'='*60}")
print(f"ABLATION AXIS 4 — Performance by conflict type")
print(f"{'='*60}")
print(f"{'Conflict Type':<22} {'N':>6} {'F1':>8} {'Accuracy':>10}")
print(f"{'-'*50}")

axis4_results = {}
for ctype in ["congruent","face-voice","face-text","voice-text","multi-conflict"]:
    subset = test_df[test_df["conflict_type"] == ctype]
    if len(subset) < 10: continue

    X_sub = np.hstack([
        np.array([parse_vec(v) for v in subset["face_vector"]]),
        np.array([parse_vec(v) for v in subset["voice_vector"]]),
        np.array([parse_vec(v) for v in subset["text_vector"]]),
        subset[["mas_face_voice","mas_face_text","mas_voice_text"]].values
    ]).astype(np.float32)

    X_sub_s = scaler.transform(X_sub)
    y_sub   = subset["incongruent"].astype(int).values

    with torch.no_grad():
        preds = (torch.sigmoid(mlp_loaded(
            torch.tensor(X_sub_s,dtype=torch.float32).to("cuda")
        )).cpu().numpy() > 0.5).astype(int)

    f1  = f1_score(y_sub, preds, zero_division=0)
    acc = (preds == y_sub).mean()
    print(f"  {ctype:<22} {len(subset):>6} {f1:>8.4f} {acc:>10.4f}")
    axis4_results[ctype] = {"n":len(subset),"f1":round(f1,4),"accuracy":round(acc,4)}

# ── Save everything ───────────────────────────────────────────────
all_ablations = {
    "axis1_modality": {
        "Face only":           {"f1":0.8220,"auroc":0.7474},
        "Voice only":          {"f1":0.8137,"auroc":0.6574},
        "Text only":           {"f1":0.7958,"auroc":0.7041},
        "Face+Voice":          {"f1":0.8275,"auroc":0.8122},
        "Face+Text":           {"f1":0.8383,"auroc":0.8297},
        "Voice+Text":          {"f1":0.8140,"auroc":0.7724},
        "All 3 modalities":    {"f1":0.9003,"auroc":0.9339},
        "All 3 + MAS (ours)":  {"f1":0.8937,"auroc":0.9203},
    },
    "axis2_finetuning": {
        "B1_vader":    0.3038, "B2_voice":  0.8055,
        "B4_majority": 0.8070, "MLP":       mlp_f1,
        "CIDF_QLoRA":  cidf_f1,
    },
    "axis3_mas": {
        "without_mas": round(f1_no_mas,4),
        "with_mas":    round(f1_mas,4),
        "delta":       round(f1_mas-f1_no_mas,4),
    },
    "axis4_conflict_type": axis4_results,
}

with open(f"{LABELS}/ablation_results.json","w") as f:
    json.dump(all_ablations, f, indent=2)

print(f"\n{'='*60}")
print(f"COMPLETE RESULTS SUMMARY")
print(f"{'='*60}")
print(f"  B1 VADER              : F1=0.3038")
print(f"  B2 Voice-only         : F1=0.8055")
print(f"  B4 Majority class     : F1=0.8070")
print(f"  MLP Fusion 24d        : F1={mlp_f1:.4f}  AUROC={mlp_auroc:.4f}")
print(f"  CIDF QLoRA (ours)     : F1={cidf_f1:.4f}  AUROC={cidf_auroc:.4f}")
print(f"\n  Fine-tuning gain      : F1={cidf_f1-mlp_f1:+.4f}  AUROC={cidf_auroc-mlp_auroc:+.4f}")
print(f"  MAS contribution      : F1={f1_mas-f1_no_mas:+.4f}")
print(f"\n✓ All ablations saved → labels/ablation_results.json")
print(f"✓ Day 5 complete")

ABLATION AXIS 4 — Performance by conflict type
Conflict Type               N       F1   Accuracy
--------------------------------------------------
  congruent                 846   0.0000     0.7187
  face-voice                499   0.9473     0.8998
  face-text                 218   0.9519     0.9083
  voice-text                239   0.9354     0.8787
  multi-conflict            813   0.9818     0.9643

COMPLETE RESULTS SUMMARY
  B1 VADER              : F1=0.3038
  B2 Voice-only         : F1=0.8055
  B4 Majority class     : F1=0.8070
  MLP Fusion 24d        : F1=0.8997  AUROC=0.9291
  CIDF QLoRA (ours)     : F1=0.9935  AUROC=0.9895

  Fine-tuning gain      : F1=+0.0938  AUROC=+0.0604
  MAS contribution      : F1=+0.0055

✓ All ablations saved → labels/ablation_results.json
✓ Day 5 complete
